# V5 Feature-Group Ablation Study (HANDOFF #3)

**Purpose:** Identify which feature groups (rolling / context / usage / advanced / opp_offense) add measurable signal to V5 predictions. Drop the groups that don't → leaner production model for Task 3.2c.

**How it works:**
1. For each (position, group) pair, retrain StatPredictors with that group's columns dropped.
2. Compute MAE delta vs Task 3.2 baseline.
3. If |delta| < 0.05 MAE → group adds no signal → drop candidate.

**Scope:** 23 ablation runs (4 groups × 5 player positions + 3 groups × 1 DST position).

**Expected runtime on Colab Pro (with eval_seasons=[2023, 2024]):** 2-4 hours.

**Resumable:** per (position, group). If `ablation_{POS}_remove_{GROUP}.json` exists in output_dir, skip.

## Step 1 — Mount Drive

In [ ]:
print('[Step 1/6] Mounting Drive...')
from google.colab import drive
drive.mount('/content/drive')
print('[Step 1/6] Drive mounted')

## Step 2 — Set paths + verify code / baseline present

In [ ]:
print('[Step 2/6] Setting paths...')
import sys
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/SportsAnalyzer'
FEATURES_DIR = f'{DRIVE_ROOT}/output/features/v5'
BASELINE_CSV = f'{DRIVE_ROOT}/output/models/v5/_mae_summary_consolidated.csv'
ABLATION_DIR = f'{DRIVE_ROOT}/output/models/v5_ablation'
CODE_ROOT = DRIVE_ROOT

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

assert Path(FEATURES_DIR).exists(), f'Features missing: {FEATURES_DIR} — run v5_feature_engineering.ipynb first'
assert Path(BASELINE_CSV).exists(), f'Baseline CSV missing: {BASELINE_CSV} — run v5_training.ipynb first'
assert Path(f'{CODE_ROOT}/src/nfl/training/v5/ablation.py').exists(), \
    f'ERROR: ablation.py not uploaded. Copy src/nfl/training/v5/ablation.py to Drive.'
assert Path(f'{CODE_ROOT}/src/nfl/features/v5/config.py').exists(), \
    f'ERROR: features/v5/config.py missing.'

for init_path in [
    f'{CODE_ROOT}/src/__init__.py',
    f'{CODE_ROOT}/src/nfl/__init__.py',
    f'{CODE_ROOT}/src/nfl/training/__init__.py',
]:
    Path(init_path).touch(exist_ok=True)

# Verify opp_offense group exists in config.py (added for Task 3.2b — if user
# forgot to re-upload config.py, this raises early instead of hours later
# when DST ablation tries to use it).
from src.nfl.features.v5.config import FEATURE_GROUP_PREFIXES
assert 'opp_offense' in FEATURE_GROUP_PREFIXES, \
    "ERROR: config.py on Drive is STALE — missing 'opp_offense' group. Re-upload src/nfl/features/v5/config.py"

Path(ABLATION_DIR).mkdir(parents=True, exist_ok=True)
print(f'[Step 2/6] Paths verified')
print(f'  Features: {FEATURES_DIR}')
print(f'  Baseline: {BASELINE_CSV}')
print(f'  Ablation output: {ABLATION_DIR}')
print(f'  FEATURE_GROUP_PREFIXES has {len(FEATURE_GROUP_PREFIXES)} groups: {list(FEATURE_GROUP_PREFIXES)}')

## Step 3 — Resume check + install ML libs

In [ ]:
print('[Step 3/6] Resume check + verifying ML libs...')
import importlib
for lib in ['xgboost', 'lightgbm', 'catboost']:
    try:
        importlib.import_module(lib)
        print(f'  OK {lib}')
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', '-q', lib], check=True)

from src.nfl.training.v5.ablation import ABLATION_GROUPS, ablation_result_complete
missing, complete = [], []
for position, groups in ABLATION_GROUPS.items():
    for group in groups:
        if ablation_result_complete(position, group, Path(ABLATION_DIR)):
            complete.append((position, group))
        else:
            missing.append((position, group))

print(f'\n  Complete: {len(complete)}/{len(missing) + len(complete)} ablation runs')
print(f'  Missing:  {len(missing)}')
for pos, grp in missing[:15]:
    print(f'    {pos:4} / remove_{grp}')
if len(missing) > 15:
    print(f'    ... +{len(missing)-15} more')

## Step 4 — Run ablations

Uses eval_seasons=[2023, 2024] by default (half the folds of Task 3.2 for ~2x speedup). Ambiguous borderline decisions can be re-run with full [2021, 2022, 2023, 2024] window later.

Each run: load features → drop group cols → walk-forward eval for each stat → compute aggregate fantasy-points delta vs baseline → assign keep/drop decision → save meta JSON. No `.joblib` files written (Task 3.2c regenerates production models).

In [ ]:
print('[Step 4/6] Running ablations (eval_seasons=[2023, 2024])...')
import time
from src.nfl.training.v5.ablation import run_all_ablations
import src.nfl.training.v5.data as v5data
v5data._features_dir = lambda: Path(FEATURES_DIR)

# Also patch compare_to_baseline to read Drive's baseline CSV
import src.nfl.training.v5.ablation as v5ab
_orig_compare = v5ab.compare_to_baseline
v5ab.compare_to_baseline = lambda result, baseline_csv_path=Path(BASELINE_CSV): _orig_compare(result, baseline_csv_path)

t0 = time.time()
summary = run_all_ablations(
    eval_seasons=[2023, 2024],
    output_dir=Path(ABLATION_DIR),
)
elapsed = time.time() - t0
print(f'\n[Step 4/6] Ablations complete in {elapsed/60:.1f} min')

## Step 5 — Summary table

In [ ]:
print('[Step 5/6] Ablation summary:')
import pandas as pd
summary = pd.read_csv(Path(ABLATION_DIR) / '_ablation_summary.csv')
summary = summary.sort_values(['position', 'group_removed'])
print('\n=== Per (position, group_removed) ===')
print(summary.to_string(index=False))

# Decision summary
print('\n=== Decision breakdown ===')
print(summary.groupby('decision').size())

# Drop candidates
drops = summary[summary['decision'] == 'drop']
if len(drops):
    print('\n=== DROP CANDIDATES (no measurable signal) ===')
    print(drops[['position', 'group_removed', 'delta']].to_string(index=False))
borderline = summary[summary['decision'] == 'borderline']
if len(borderline):
    print('\n=== BORDERLINE (re-run on full 2021-2024 for confidence) ===')
    print(borderline[['position', 'group_removed', 'delta']].to_string(index=False))

## Step 6 — Verify file counts + report back to Claude

In [ ]:
print('[Step 6/6] Verification:')
from pathlib import Path
json_files = list(Path(ABLATION_DIR).glob('ablation_*_remove_*.json'))
print(f'  Ablation JSONs: {len(json_files)}  (expect 23)')
print(f'  Summary CSV:    {(Path(ABLATION_DIR) / "_ablation_summary.csv").exists()}')

print('\nReport back to Claude:')
print('  1. Paste the summary table from Step 5')
print('  2. Any KEEP/DROP/BORDERLINE counts')
print('  3. Any ablation runs that failed (missing JSON)')